# P1 · Dữ liệu & điều phối — Bước 1 & 4

**Input (cell đầu tự nạp):** data/raw/*.csv (M5 thô)

**Output (cell cuối tự lưu):** common/{sample_ids.csv, segment_labels.parquet, rolling_windows.csv} + data/processed/data_long.parquet

**Code mẫu tham khảo:** `notebooks/M5_reference.ipynb` phần **§1**.

> Chạy được **cell đầu** = đã đủ input để làm phần của bạn. Viết code ở khu vực giữa. Cuối cùng chạy **cell lưu output**.

In [ ]:
# ===== CELL ĐẦU: import thư viện + nạp INPUT (dữ liệu M5 thô) =====
import numpy as np, pandas as pd
import yaml
from pathlib import Path

CFG = yaml.safe_load(open("common/config.yaml", encoding="utf-8"))
SEED = CFG["seed"]; H = CFG["horizon"]; Q = CFG["quantiles"]
SP = CFG["split"]; RL = CFG["rolling"]; SBC = CFG["sbc"]; SAMP = CFG["sampling"]
ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

RAW = Path("data/raw")
assert (RAW / "sales_train_evaluation.csv").exists(), \
    "Chưa có dữ liệu! Đặt các file M5 vào data/raw/ (calendar, sell_prices, sales_train_evaluation)."
calendar = pd.read_csv(RAW / "calendar.csv")
prices   = pd.read_csv(RAW / "sell_prices.csv")
sales    = pd.read_csv(RAW / "sales_train_evaluation.csv")   # d_1..d_1941
D_COLS = [c for c in sales.columns if c.startswith("d_")]
train_days = [f"d_{i}" for i in range(1, SP["train_end"] + 1)]
print("✅ Input sẵn sàng — sales", sales.shape, "| calendar", calendar.shape, "| prices", prices.shape)

## Xử lý (bước 1 & 4)

In [ ]:
# --- ADI & CV^2 trên train (vectorised) ---
mat = sales[train_days].to_numpy(dtype=np.float32); T = mat.shape[1]
nz = mat > 0; n_nz = nz.sum(1)
mean_nz = np.divide(np.where(nz, mat, 0).sum(1), n_nz, out=np.zeros(len(mat)), where=n_nz > 0)
sq = np.where(nz, mat**2, 0).sum(1)
var_nz = np.clip(np.divide(sq, n_nz, out=np.zeros(len(mat)), where=n_nz > 0) - mean_nz**2, 0, None)
with np.errstate(divide="ignore", invalid="ignore"):
    ADI = np.where(n_nz > 0, T / n_nz, np.inf)
    CV2 = np.where(mean_nz > 0, (np.sqrt(var_nz) / mean_nz) ** 2, 0.0)
seg = sales[ID_COLS].copy(); seg["series_id"] = seg["id"]
seg["zero_rate"] = 1 - n_nz / T; seg["ADI"] = ADI; seg["CV2"] = CV2

In [ ]:
# --- Nhãn SBC (4 nhóm) ---
cond = [(seg.ADI < SBC["adi_threshold"]) & (seg.CV2 < SBC["cv2_threshold"]),
        (seg.ADI < SBC["adi_threshold"]) & (seg.CV2 >= SBC["cv2_threshold"]),
        (seg.ADI >= SBC["adi_threshold"]) & (seg.CV2 < SBC["cv2_threshold"]),
        (seg.ADI >= SBC["adi_threshold"]) & (seg.CV2 >= SBC["cv2_threshold"])]
seg["segment"] = np.select(cond, ["smooth","erratic","intermittent","lumpy"], default="intermittent")
print(seg.segment.value_counts().to_dict())

In [ ]:
# --- Mẫu phân tầng: toàn bộ smooth+erratic, cap intermittent+lumpy ---
rare = {"smooth","erratic"}; cap = SAMP["cap"]; picks = []
for s, g in seg.groupby("segment"):
    n = len(g) if s in rare else min(len(g), cap)
    picks.append(g.sample(n=n, random_state=SEED))
sample_ids = sorted(pd.concat(picks)["series_id"].tolist())
print("Mẫu:", len(sample_ids), "chuỗi")

In [ ]:
# --- Bước 1: wide -> long (chỉ chuỗi mẫu) + join calendar/giá + feature ---
CAL = ["d","date","wm_yr_wk","wday","month","year","event_name_1","event_type_1","snap_CA","snap_TX","snap_WI"]
sub = sales[sales.id.isin(sample_ids)]
long = sub.melt(id_vars=ID_COLS, value_vars=D_COLS, var_name="d", value_name="sales")
long = long.merge(calendar[CAL], on="d", how="left").merge(prices, on=["store_id","item_id","wm_yr_wk"], how="left")
long["series_id"] = long["id"]; long["date"] = pd.to_datetime(long["date"])
long["d_int"] = long["d"].str.slice(2).astype("int32")
long["is_event"] = long["event_name_1"].notna().astype("int8")
long["snap"] = 0
for st, c in {"CA":"snap_CA","TX":"snap_TX","WI":"snap_WI"}.items():
    m = long.state_id == st; long.loc[m, "snap"] = long.loc[m, c].values
long["snap"] = long["snap"].astype("int8"); long["sales"] = long["sales"].astype("float32")
long = long.merge(seg[["series_id","segment"]], on="series_id", how="left")
long = long.sort_values(["series_id","d_int"]).reset_index(drop=True)
print("data_long:", long.shape)

In [ ]:
# --- Bước 4: cửa sổ rolling-origin (>=3), tất cả <= d_1941 ---
val_len = SP["val_end"] - SP["train_end"]; rows = []
for k in range(RL["n_windows"]):
    te = SP["test_end"] - k * RL["step"]; ts = te - H + 1; ve = ts - 1; tr = ve - val_len
    assert tr >= 1 and te <= SP["max_actual_day"] and te - ts + 1 == H
    rows.append(dict(window=f"W{k+1}", train_end=tr, val_end=ve, test_start=ts, test_end=te))
windows = pd.DataFrame(rows); print(windows.to_string(index=False))

In [ ]:
# ===== CELL CUỐI: lưu OUTPUT (đừng sửa format — cả nhóm phụ thuộc) =====
Path("common").mkdir(exist_ok=True); Path("data/processed").mkdir(parents=True, exist_ok=True)
seg[["series_id","ADI","CV2","zero_rate","segment"]].to_parquet("common/segment_labels.parquet", index=False)
pd.DataFrame({"series_id": sample_ids}).to_csv("common/sample_ids.csv", index=False)
windows.to_csv("common/rolling_windows.csv", index=False)
long.to_parquet("data/processed/data_long.parquet", index=False)
print("✅ Đã lưu: common/{segment_labels,sample_ids,rolling_windows} + data/processed/data_long.parquet")